# 02. Подготовка признаков

**Цель:** сравнить native-категории CatBoost и ordered target encoding (OTE).
**Условия:** временные доли 70/10/5/15%; история исключает текущий timestamp-пакет. OTE предполагает немедленную доступность прошлых меток.

[Итоговый отчёт](../reports/full/RESULTS.md)

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Запустите notebook из корня проекта или notebooks")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import uuid
from aml.pipeline import (TARGET, OrderedTargetEncoder, basic_features, historical_features,
    load_raw, temporal_slices, ranking_metrics, new_run, set_active)
from sklearn.metrics import average_precision_score
RUN_DIR = new_run(PROJECT_ROOT / "artifacts", "runs")
DATA_DIR = RUN_DIR / "data"
DATA_DIR.mkdir()
SEED = 42

In [2]:
df = load_raw(PROJECT_ROOT)
timestamp = df["Timestamp"]
y = df[TARGET]
periods = temporal_slices(timestamp)
X = historical_features(basic_features(df), timestamp)
del df
summary = pd.DataFrame({name: {"rows": len(y.iloc[s]), "positives": int(y.iloc[s].sum()),
    "start": timestamp.iloc[s].min(), "end": timestamp.iloc[s].max()} for name,s in periods.items()}).T
display(summary)
for name in ("train", "val", "threshold"):
    if y.iloc[periods[name]].nunique() != 2:
        raise ValueError(f"{name}: нужны оба класса; увеличьте период данных")
X_train, X_val = (X.iloc[periods[k]].reset_index(drop=True) for k in ("train", "val"))
y_train, y_val = (y.iloc[periods[k]].reset_index(drop=True) for k in ("train", "val"))
timestamp_train = timestamp.iloc[periods["train"]].reset_index(drop=True)

,rows,positives,start,end
train,4846503,2231,2022-09-01 00:00:00,2022-09-07 14:47:00
val,692487,409,2022-09-07 14:48:00,2022-09-08 16:05:00
threshold,345949,174,2022-09-08 16:06:00,2022-09-09 03:12:00
test,1039102,751,2022-09-09 03:13:00,2022-09-17 15:28:00


## Выборка для сравнения

In [3]:
y_array = y_train.to_numpy()
positive_positions = np.flatnonzero(y_array == 1)
negative_positions = np.flatnonzero(y_array == 0)

rng = np.random.default_rng(42)
negative_sample_size = max(1, int(0.3 * len(negative_positions)))
sampled_negative_positions = rng.choice(
    negative_positions,
    size=negative_sample_size,
    replace=False,
)

# Both models see the same rows in the same chronological order.
fit_positions = np.sort(
    np.concatenate([positive_positions, sampled_negative_positions])
)

X_fit = X_train.iloc[fit_positions].reset_index(drop=True)
y_fit = y_train.iloc[fit_positions].reset_index(drop=True)
timestamp_fit = timestamp_train.iloc[fit_positions].reset_index(drop=True)

negative_weight = len(negative_positions) / negative_sample_size
sample_weight = np.where(
    y_fit.to_numpy() == 0,
    negative_weight,
    1.0,
).astype("float64")

assert timestamp_fit.is_monotonic_increasing
assert len(X_fit) == len(y_fit) == len(timestamp_fit) == len(sample_weight)

pd.Series(
    {
        "fit_rows": len(X_fit),
        "fit_positives": int(y_fit.sum()),
        "negative_sampling_rate": negative_sample_size / len(negative_positions),
        "negative_weight": negative_weight,
    },
    name="training subset",
)

fit_rows                  1.455512e+06
fit_positives             2.231000e+03
negative_sampling_rate    2.999999e-01
negative_weight           3.333335e+00
Name: training subset, dtype: float64

## Native CatBoost

`has_time=True` сохраняет порядок, но не гарантирует исключения timestamp-пакета из внутренних CTR.

In [4]:
from catboost import CatBoostClassifier
categorical_features = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
]

missing_categorical = set(categorical_features) - set(X_fit.columns)
assert not missing_categorical, f"Categorical columns missing: {sorted(missing_categorical)}"

model_params = dict(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=6.0,
    random_seed=42,
    has_time=True,
    allow_writing_files=False,
    thread_count=4,
)

cat_native = CatBoostClassifier(**model_params)
cat_native.fit(
    X_fit,
    y_fit,
    sample_weight=sample_weight,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    cat_features=categorical_features,
    verbose=False,
)

native_val_score = cat_native.predict_proba(X_val)[:, 1]
catboost_native_pr_auc = average_precision_score(y_val, native_val_score)
catboost_native_pr_auc

0.26396073451691987

## Ordered target encoding

In [5]:
target_encoding_interactions = [
    ("From Bank", "Account"),
    ("To Bank", "Account.1"),
    ("From Bank", "To Bank"),
    ("Receiving Currency", "Payment Currency"),
]

ordered_target_encoder = OrderedTargetEncoder(
    categorical_features=categorical_features,
    interactions=target_encoding_interactions,
    smoothing=(20.0, 200.0),
    add_log_count=True,
)

In [6]:
X_fit_ote = ordered_target_encoder.fit_transform(
    X_fit,
    y_fit,
    timestamp_fit,
    sample_weight=sample_weight,
)
X_val_ote = ordered_target_encoder.transform(X_val)

non_numeric_ote = X_fit_ote.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
assert not non_numeric_ote, f"OTE output must be numeric, found: {non_numeric_ote}"
assert list(X_fit_ote.columns) == list(X_val_ote.columns)

X_fit_ote.shape, X_val_ote.shape

((1455512, 65), (692487, 65))

In [7]:
cat_ote = CatBoostClassifier(**model_params)
cat_ote.fit(
    X_fit_ote,
    y_fit,
    sample_weight=sample_weight,
    eval_set=(X_val_ote, y_val),
    early_stopping_rounds=150,
    verbose=False,
)

ote_val_score = cat_ote.predict_proba(X_val_ote)[:, 1]
ordered_te_pr_auc = average_precision_score(y_val, ote_val_score)
ordered_te_pr_auc

0.29638966959617613

## Сравнение на validation

In [8]:
comparison = pd.DataFrame({"native": ranking_metrics(y_val, native_val_score),
                           "OTE": ranking_metrics(y_val, ote_val_score)}).T
comparison.to_csv(RUN_DIR / "encoding_comparison.csv")
display(comparison)

,AP,ROC-AUC
native,0.263961,0.976843
OTE,0.296390,0.975759


## Сохранение признаков

Для notebook 03 encoder обучается на полном train. Сохраняются splits, encoder и схема; test здесь не оценивается.

In [9]:
final_encoder = OrderedTargetEncoder(categorical_features, target_encoding_interactions)
X_train_ote = final_encoder.fit_transform(X_train, y_train, timestamp_train)
run_id = str(uuid.uuid4())
for name, s in periods.items():
    encoded = X_train_ote if name == "train" else final_encoder.transform(X.iloc[s])
    if not np.isfinite(encoded.to_numpy()).all():
        raise ValueError(f"Nonfinite features: {name}")
    encoded[TARGET] = y.iloc[s].to_numpy()
    encoded.to_parquet(DATA_DIR / f"{name}.parquet", index=False)
    timestamp.iloc[s].to_frame().to_parquet(DATA_DIR / f"{name}_time.parquet", index=False)
joblib.dump(final_encoder, RUN_DIR / "encoder.joblib")
metadata = {"run_id": run_id, "seed": SEED,
            "features": [c for c in X_train_ote.columns if c != TARGET],
            "periods": summary.astype(str).to_dict(orient="index"),
            "label_assumption": "labels available immediately; offline only"}
(RUN_DIR / "data_manifest.json").write_text(json.dumps(metadata, indent=2))
print("Saved:", DATA_DIR)
set_active(PROJECT_ROOT / "artifacts/runs", "active.json", RUN_DIR)

Saved: artifacts/runs/full/data


## Выводы

- OTE повысил validation AP с 0,2640 до 0,2964: +0,0324 в этом сравнении.
- Для notebook 03 OTE переобучается на полном train; его результаты не являются прямым продолжением sampled-сравнения.
- Время фактического получения меток неизвестно: пригодность OTE для реального скоринга пока не подтверждена.